In [1]:
import torch
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from torch import nn, optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score, precision_recall_fscore_support
import itertools
import torch
from efficientnet_pytorch import EfficientNet

# Set the device to GPU if available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [2]:
data_dir = 'Transformed_Image'

In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [4]:
dataset = datasets.ImageFolder(data_dir,transform=transform)

In [5]:
train_size = int(0.8 * len(dataset))  # 80% of dataset
val_size = len(dataset) - train_size  # 20% of dataset
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [6]:
print(train_size,val_size)

15001 3751


In [6]:
trainloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valloader = DataLoader(val_dataset, batch_size=32, shuffle=True)

In [7]:
# Load a pre-trained model (e.g., ResNet18) and modify it for our task
model = models.resnet50(pretrained=True)

# Freeze all the layers in the pretrained model
for param in model.parameters():
    param.requires_grad = False

# Replace the final layer with a new one that matches our number of classes (4 classes in our case)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 4) 

# Move the model to the GPU if available
model = model.to(device)

c:\Users\Aishu\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Aishu\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [8]:
# Define the loss function
criterion = nn.CrossEntropyLoss()

# Define the optimizer, only optimizing the parameters of the final layer
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

In [10]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    running_loss = 0.0
    
    for inputs, labels in trainloader:
        # Move the input and label tensors to the correct device
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Zero the parameter gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    # Print statistics
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(trainloader)}")
    
    # Validation loss
    model.eval()  # Set the model to evaluation mode
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in valloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    print(f'Validation Loss: {val_loss / len(valloader)}, Accuracy: {100 * correct / total}%')
torch.save(model.state_dict(), 'models/cnn_model_pred.pth')
  

Epoch 1, Loss: 0.9646913024154045
Validation Loss: 0.9457397314451509, Accuracy: 56.544921354305515%
Epoch 2, Loss: 0.9181737599850718
Validation Loss: 0.8661144103034067, Accuracy: 60.01066382298054%
Epoch 3, Loss: 0.8803209808589553
Validation Loss: 0.8669629677877588, Accuracy: 60.43721674220208%
Epoch 4, Loss: 0.8703457953324959
Validation Loss: 0.9232668735213199, Accuracy: 56.43828312450013%
Epoch 5, Loss: 0.8657215521025505
Validation Loss: 0.9348591955031379, Accuracy: 54.998667022127435%
Epoch 6, Loss: 0.8597799713677688
Validation Loss: 0.8689966883699772, Accuracy: 61.15702479338843%
Epoch 7, Loss: 0.8457922676224698
Validation Loss: 0.8455074868969998, Accuracy: 61.07704612103439%
Epoch 8, Loss: 0.8406444605606705
Validation Loss: 0.867735122725115, Accuracy: 58.67768595041322%
Epoch 9, Loss: 0.8350829858261385
Validation Loss: 0.8748817453950138, Accuracy: 59.904025593175156%
Epoch 10, Loss: 0.8359336222666921
Validation Loss: 0.8419634053262613, Accuracy: 61.4502799253532

In [9]:
efficientnet_model = EfficientNet.from_pretrained('efficientnet-b0', num_classes=4)

# Move the model to the GPU if available
efficientnet_model = efficientnet_model.to(device)

Loaded pretrained weights for efficientnet-b0


In [10]:
optimizer = optim.Adam(efficientnet_model.parameters(), lr=0.001)

In [11]:
num_epochs = 10

# train efficientnet model
for epoch in range(num_epochs):
    efficientnet_model.train()  # Set the model to training mode
    running_loss = 0.0
    
    for inputs, labels in trainloader:
        # Move the input and label tensors to the correct device
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Zero the parameter gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = efficientnet_model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    # Print statistics
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(trainloader)}")
    
    # Validation loss
    efficientnet_model.eval()  # Set the model to evaluation mode
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in valloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = efficientnet_model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    print(f'Validation Loss: {val_loss / len(valloader)}, Accuracy: {100 * correct / total}%')

# Save the model
    
torch.save(efficientnet_model.state_dict(),'models/eff_model_pred.pth')

Epoch 1, Loss: 0.7659876162309383
Validation Loss: 1.1288237005977306, Accuracy: 61.42362036790189%
Epoch 2, Loss: 0.4010763830944165
Validation Loss: 0.4627701763253091, Accuracy: 81.44494801386297%
Epoch 3, Loss: 0.2154993923313455
Validation Loss: 0.296665665804077, Accuracy: 89.1229005598507%
Epoch 4, Loss: 0.1331751895691159
Validation Loss: 0.35457099933903347, Accuracy: 87.44334844041589%
Epoch 5, Loss: 0.1072815011666496
Validation Loss: 0.30995676977465214, Accuracy: 89.94934684084244%
Epoch 6, Loss: 0.07032204443302903
Validation Loss: 0.3550501326099038, Accuracy: 88.45641162356705%
Epoch 7, Loss: 0.0707315684741498
Validation Loss: 0.1952437828689561, Accuracy: 93.148493735004%
Epoch 8, Loss: 0.06340079829367354
Validation Loss: 0.23572907558942247, Accuracy: 92.37536656891496%
Epoch 9, Loss: 0.059935887533635684
Validation Loss: 0.34846646429169764, Accuracy: 90.13596374300187%
Epoch 10, Loss: 0.05346272350139995
Validation Loss: 0.2171265145613796, Accuracy: 92.8552386030

In [ ]:
alexnet_model = models.alexnet(pretrained=True)

c:\Users\Aishu\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
alexnet_model.classifier[6] = nn.Linear(alexnet_model.classifier[6].in_features, 4)

In [ ]:
optimizer = optim.Adam(alexnet_model.parameters(), lr=0.001)

In [ ]:
alexnet_model = alexnet_model.to(device)

In [ ]:
num_epochs = 10

In [ ]:
for epoch in range(num_epochs):
    alexnet_model.train()  # Set the model to training mode
    running_loss = 0.0

    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = alexnet_model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(trainloader):.4f}")
    alexnet_model.eval() 
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in valloader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = alexnet_model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_accuracy = correct / total
    print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")
torch.save(alexnet_model.state_dict(), 'models/alexnet_model.pth')


Epoch 1/10, Loss: 1.1522
Validation Accuracy: 44.86%
Epoch 2/10, Loss: 1.0843
Validation Accuracy: 44.86%
Epoch 3/10, Loss: 1.0851
Validation Accuracy: 44.86%
Epoch 4/10, Loss: 1.0825
Validation Accuracy: 44.86%
Epoch 5/10, Loss: 1.0815
Validation Accuracy: 44.86%
Epoch 6/10, Loss: 1.0806
Validation Accuracy: 44.86%
Epoch 7/10, Loss: 1.0834
Validation Accuracy: 40.77%
Epoch 8/10, Loss: 1.0810
Validation Accuracy: 44.86%
Epoch 9/10, Loss: 1.0798
Validation Accuracy: 44.86%
Epoch 10/10, Loss: 1.0788
Validation Accuracy: 44.86%
